# MarketSense: Do Small AI Models Understand Financial News?
**An Empirical Study of Compact Language Models and Stock Return Reactions**

**Author:** Sai Mahesh Sandeboina — Pace University, USA  
**Date:** October 2025

---

## Reproducibility Instructions

This notebook reproduces **all figures and statistics** reported in the paper.

**Required files in your Colab session (upload before running):**
- `headlines_scored.csv` — headlines with FinBERT and DistilBERT scores
- `prices.csv` — daily OHLC prices from Stooq (Apr–Oct 2025)

> ⚠️ Do NOT re-scrape live data. Results will differ from the paper due to feed changes.

**Run cells in order: 1 → 2 → 3 → 4 → 5 → 6**

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────────────
!pip install -q \
    pandas==2.2.2 \
    numpy==1.26.4 \
    scipy==1.11.4 \
    matplotlib==3.9.0 \
    seaborn==0.13.2 \
    transformers==4.41.2 \
    torch==2.3.1 \
    tokenizers==0.19.1 \
    feedparser==6.0.10 \
    pandas-datareader==0.10.0 \
    tqdm==4.66.5

print('✅ All dependencies installed.')

In [ ]:
# ── Cell 2: Imports & Configuration ───────────────────────────────────────────
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)

TICKERS = ['AAPL', 'AMZN', 'GS', 'JPM', 'MS', 'MSFT', 'NVDA', 'TSLA', 'WMT']

COLORS = {
    'finbert':  '#E07B00',   # orange  — matches paper figures
    'distil':   '#2166AC',   # blue    — matches paper figures
    'scatter':  '#5BA4CF',
}

print('✅ Imports complete.')

In [ ]:
# ── Cell 3: Load Data & Compute Daily Sentiment ────────────────────────────────
#
# Reads from saved CSVs — reproduces exact paper results.

# --- Load scored headlines ---
scored = pd.read_csv('headlines_scored.csv')
scored['date'] = pd.to_datetime(scored['published_utc']).dt.date.astype(str)

# --- Daily average sentiment per (date, ticker) ---
daily = (
    scored
    .groupby(['date', 'ticker'], as_index=False)
    .agg(
        finbert_sent=('finbert_sent', 'mean'),
        distil_sent=('distil_sent',  'mean')
    )
    .sort_values(['date', 'ticker'])
)

# --- Load prices, compute open→close return ---
prices = pd.read_csv('prices.csv')
prices['date']   = prices['date'].astype(str)
prices['ticker'] = prices['ticker'].astype(str)
prices['Open']   = pd.to_numeric(prices['Open'],  errors='coerce')
prices['Close']  = pd.to_numeric(prices['Close'], errors='coerce')
prices['ret']    = (prices['Close'] - prices['Open']) / prices['Open']

# --- Inner join: align sentiment with returns ---
merged = daily.merge(
    prices[['date', 'ticker', 'ret']],
    on=['date', 'ticker'],
    how='inner'
).dropna(subset=['finbert_sent', 'distil_sent', 'ret'])

merged.to_csv('merged_sentiment_returns.csv', index=False)

print(f'✅ Loaded {len(merged)} aligned (date, ticker) observations')
print(f'   Tickers: {sorted(merged["ticker"].unique())}')
print(f'   Date range: {merged["date"].min()} → {merged["date"].max()}')

In [ ]:
# ── Cell 4: Correlation Analysis ──────────────────────────────────────────────
#
# Computes Pearson r, p-value, and bootstrapped 95% CIs.
# Reproduces Table 1 from the paper.

def pearson_with_ci(x, y, n_boot=2000, seed=42):
    r, p = pearsonr(x, y)
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        xb, yb = x[idx], y[idx]
        if xb.std() > 0 and yb.std() > 0:
            boot.append(pearsonr(xb, yb)[0])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return r, p, lo, hi

results = []
df = merged.copy()

# Overall pooled
for model, col in [('FinBERT', 'finbert_sent'), ('DistilBERT', 'distil_sent')]:
    r, p, lo, hi = pearson_with_ci(df[col].values, df['ret'].values)
    results.append({'Ticker': 'ALL', 'Model': model,
                    'r': r, 'p': p, 'CI_lo': lo, 'CI_hi': hi, 'n': len(df)})

# Per-ticker
for ticker in TICKERS:
    sub = df[df['ticker'] == ticker]
    if len(sub) < 10:
        continue
    for model, col in [('FinBERT', 'finbert_sent'), ('DistilBERT', 'distil_sent')]:
        r, p, lo, hi = pearson_with_ci(sub[col].values, sub['ret'].values)
        results.append({'Ticker': ticker, 'Model': model,
                        'r': r, 'p': p, 'CI_lo': lo, 'CI_hi': hi, 'n': len(sub)})

corr_df = pd.DataFrame(results)
corr_df.to_csv('correlation_results.csv', index=False)

# Print summary
print('='*65)
print('CORRELATION SUMMARY — Reproduces Paper Results')
print('='*65)
overall = corr_df[corr_df['Ticker'] == 'ALL']
for _, row in overall.iterrows():
    print(f"  {row['Model']:12s}  r={row['r']:+.3f}  p={row['p']:.4f}  "
          f"95% CI=[{row['CI_lo']:+.3f}, {row['CI_hi']:+.3f}]  n={int(row['n'])}")
print('-'*65)
pivot = corr_df[corr_df['Ticker'] != 'ALL'].pivot(
    index='Ticker', columns='Model', values='r'
).reindex(TICKERS)
print(pivot.round(3).to_string())
print('='*65)

In [ ]:
# ── Cell 5: Generate All Figures ──────────────────────────────────────────────
#
# Reproduces Figures 1–6 from the paper.

df     = merged.copy()
pivot  = corr_df[corr_df['Ticker'] != 'ALL'].pivot(
    index='Ticker', columns='Model', values='r'
).reindex(TICKERS)

# ── Figure 1: FinBERT Pooled Scatter ──────────────────────────────────────────
def pooled_scatter(xcol, model_name, color, fig_num):
    row = corr_df[(corr_df['Ticker'] == 'ALL') & (corr_df['Model'] == model_name)].iloc[0]
    x, y = df[xcol].values, df['ret'].values
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(x, y, alpha=0.55, s=35, color=COLORS['scatter'], edgecolors='none')
    m, b = np.polyfit(x, y, 1)
    xl = np.linspace(x.min(), x.max(), 200)
    ax.plot(xl, m*xl+b, color=color, linewidth=2)
    ax.axhline(0, color='gray', lw=0.6, ls='--')
    ax.axvline(0, color='gray', lw=0.6, ls='--')
    ax.set_xlabel(f'{xcol} sentiment', fontsize=11)
    ax.set_ylabel('Same-day return (Open→Close)', fontsize=11)
    ax.set_title(
        f'Overall {model_name} Sentiment vs Same-Day Return\n'
        f"r={row['r']:.3f}, p={row['p']:.4f}, n={int(row['n'])}",
        fontsize=12
    )
    sns.despine(ax=ax)
    fig.tight_layout()
    path = f'figures/fig{fig_num}_{model_name.lower()}_pooled_scatter.png'
    fig.savefig(path, dpi=150)
    plt.show()
    print(f'✅ Saved {path}')

pooled_scatter('finbert_sent', 'FinBERT',    COLORS['finbert'], 1)
pooled_scatter('distil_sent',  'DistilBERT', COLORS['distil'],  2)

# ── Figure 3: Per-Ticker Bar Chart ────────────────────────────────────────────
x = np.arange(len(TICKERS))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, pivot['DistilBERT'], w, label='DistilBERT',
       color=COLORS['distil'],  alpha=0.85)
ax.bar(x + w/2, pivot['FinBERT'],    w, label='FinBERT',
       color=COLORS['finbert'], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(TICKERS, fontsize=10)
ax.set_ylabel('Pearson r', fontsize=11)
ax.set_xlabel('Ticker', fontsize=11)
ax.set_title('Per-Ticker Pearson Correlation (Sentiment vs Same-Day Return)', fontsize=12)
ax.axhline(0, color='black', lw=0.8)
ax.legend(fontsize=10)
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig('figures/fig3_per_ticker_correlation.png', dpi=150)
plt.show()
print('✅ Saved figures/fig3_per_ticker_correlation.png')

# ── Figure 4: 3-Day Rolling Sentiment ────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for i, ticker in enumerate(TICKERS):
    sub = df[df['ticker'] == ticker].sort_values('date').copy()
    sub['date'] = pd.to_datetime(sub['date'])
    sub['fb_roll'] = sub['finbert_sent'].rolling(3, min_periods=1).mean()
    sub['db_roll'] = sub['distil_sent'].rolling(3,  min_periods=1).mean()
    ax = axes.flatten()[i]
    ax.plot(sub['date'], sub['fb_roll'], color=COLORS['finbert'],
            label='FinBERT (3d avg)', lw=1.5)
    ax.plot(sub['date'], sub['db_roll'], color=COLORS['distil'],
            label='DistilBERT (3d avg)', lw=1.2, ls='--')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_title(f'3-Day Rolling Sentiment — {ticker}', fontsize=9)
    ax.tick_params(axis='x', labelsize=6, rotation=30)
    ax.tick_params(axis='y', labelsize=7)
    if i == 0:
        ax.legend(fontsize=7)
fig.suptitle('FinBERT vs DistilBERT: 3-Day Rolling Sentiment by Ticker', fontsize=12)
fig.tight_layout()
fig.savefig('figures/fig4_rolling_sentiment.png', dpi=150)
plt.show()
print('✅ Saved figures/fig4_rolling_sentiment.png')

# ── Figures 5 & 6: Per-Ticker Scatter (AAPL, MSFT) ───────────────────────────
def per_ticker_scatter(ticker, fig_num):
    sub = df[df['ticker'] == ticker]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, (model, col, color) in zip(axes, [
        ('FinBERT',    'finbert_sent', COLORS['finbert']),
        ('DistilBERT', 'distil_sent',  COLORS['distil']),
    ]):
        row = corr_df[(corr_df['Ticker'] == ticker) &
                      (corr_df['Model']  == model)].iloc[0]
        x, y = sub[col].values, sub['ret'].values
        ax.scatter(x, y, alpha=0.55, s=35,
                   color=COLORS['scatter'], edgecolors='none')
        m, b = np.polyfit(x, y, 1)
        xl = np.linspace(x.min(), x.max(), 200)
        ax.plot(xl, m*xl+b, color=color, lw=2)
        ax.set_xlabel('Daily sentiment', fontsize=10)
        ax.set_ylabel('Open→Close return', fontsize=10)
        ax.set_title(
            f"{ticker} — {model} Sentiment vs Return\n"
            f"r={row['r']:.3f}, p={row['p']:.4f}, n={int(row['n'])}",
            fontsize=10
        )
        ax.axhline(0, color='gray', lw=0.5, ls='--')
        sns.despine(ax=ax)
    fig.suptitle(
        f'{ticker}: FinBERT (left) vs DistilBERT (right) sentiment vs return',
        fontsize=11
    )
    fig.tight_layout()
    path = f'figures/fig{fig_num}_{ticker.lower()}_scatter.png'
    fig.savefig(path, dpi=150)
    plt.show()
    print(f'✅ Saved {path}')

per_ticker_scatter('AAPL', 5)
per_ticker_scatter('MSFT', 6)

print('\n✅ All figures generated and saved to ./figures/')

In [ ]:
# ── Cell 6: Download All Results ──────────────────────────────────────────────
import zipfile
from google.colab import files

with zipfile.ZipFile('marketsense_results.zip', 'w') as z:
    for f in ['merged_sentiment_returns.csv',
               'correlation_results.csv',
               'daily_sentiment.csv']:
        if os.path.exists(f):
            z.write(f)
    for fig_file in os.listdir('figures'):
        z.write(f'figures/{fig_file}')

files.download('marketsense_results.zip')
print('✅ Downloaded marketsense_results.zip')